In [ ]:
distance = get_distance(coord)
adjacency_list = np.array([np.where(neigh)[0] for neigh in adjacency_matrix], dtype=object)
N = adjacency_matrix.shape[0]
num_grs_iter = int(np.ceil(np.log2(N-1)))
gcn = greedy_closest_neighbour(adjacency_list, distance)
gd = greedy_destination(gcn.copy(), num_grs_iter)
gd = drip_down(gcn, gd, distance)
clogging_count = np.bincount(gd.flatten()) - np.sum(gd == np.arange(N), axis=0)

In [ ]:
#adjacency_matrix, coord = graphs['pso_512']
adjacency_matrix, coord = graphs['pso_128']
adjacency_list = np.array([np.where(neigh)[0] for neigh in adjacency_matrix], dtype=object)
N = adjacency_matrix.shape[0]
arange = np.arange(N)
num_grs_iter = int(np.ceil(np.log2(N-1)))

#coord = get_random_init(adjacency_matrix)


dist = get_distance(coord)
gcn = greedy_closest_neighbour(adjacency_list, dist)
gd = greedy_destination(gcn.copy(), num_grs_iter)
grs = greedy_routing_score(gd)

In [ ]:
moved_vertex = np.random.randint(N)
new_coord = random_move(coord.copy(), moved_vertex)
new_dist = get_distance(new_coord)
new_gcn_alt = greedy_closest_neighbour(adjacency_list, new_dist)
new_gd_alt = greedy_destination(new_gcn_alt.copy(), num_grs_iter)

In [ ]:
distance = new_dist.copy()
moved_vertex_neighbours = adjacency_list[moved_vertex]
moved_vertex, moved_vertex_neighbours

In [ ]:
def recalc_greedy_routing(adjacency_list, distance, num_grs_iter, gcn, gd, moved_vertex):
    N = adjacency_list.shape[0]
    
    affected = np.append(adjacency_list[moved_vertex], moved_vertex)
    affected_next_hop = np.array([neighbours[np.argmin(distance[neighbours], axis=0)] for neighbours in adjacency_list[affected]])
    affected_next_hop[np.arange(affected.shape[0]), affected] = affected
    changed = np.any(affected_next_hop != gcn[affected], axis=0)
    new_gcn = gcn.copy()
    
    new_gcn[:, moved_vertex] = np.array([neighbours[np.argmin(distance[neighbours, moved_vertex], axis=0)] for neighbours in adjacency_list])
    new_gcn[affected] = affected_next_hop
    
    changed[moved_vertex] = True
    new_gd = gd.copy()
    new_gd[:, changed] = greedy_destination(new_gcn[:, changed], num_grs_iter)
    return new_gcn, new_gd

In [ ]:
affected = np.append(adjacency_list[moved_vertex], moved_vertex)
affected_next_hop = np.array([neighbours[np.argmin(distance[neighbours], axis=0)] for neighbours in adjacency_list[affected]])
affected_next_hop[np.arange(affected.shape[0]), affected] = affected

new_gcn = gcn.copy()
new_gcn[:, moved_vertex] = np.array([neighbours[np.argmin(distance[neighbours, moved_vertex], axis=0)] for neighbours in adjacency_list])
new_gcn[affected] = affected_next_hop

In [ ]:
prev_affected_next_hop = gcn[affected]

In [ ]:
changed = affected_next_hop != prev_affected_next_hop

In [ ]:
source, target = np.where(changed)
source = affected[source]

In [ ]:


def recalc_greedy_routing(adjacency_list, distance, num_grs_iter, gcn, gd, moved_vertex):
    N = adjacency_list.shape[0]
    moved_vertex_neighbours = adjacency_list[moved_vertex]

    arange = np.arange(N)
    moved_vertex_neighbours_next_hop = np.array([neighbours[np.argmin(distance[neighbours], axis=0)] for neighbours in adjacency_list[moved_vertex_neighbours]])
    moved_vertex_neighbours_next_hop[np.arange(moved_vertex_neighbours.shape[0]), moved_vertex_neighbours] = moved_vertex_neighbours

    moved_vertex_next_hop = np.argmin(distance[moved_vertex_neighbours], axis=0)

    prev_next_next_hop = gcn[gcn[moved_vertex], arange]
    cur_next_next_hop = moved_vertex_neighbours_next_hop[moved_vertex_next_hop, arange]
    changed = prev_next_next_hop != cur_next_next_hop
    changed_too = np.any(gcn[moved_vertex_neighbours] != moved_vertex_neighbours_next_hop, axis=0)
    changed = changed | changed_too

    new_gcn = gcn.copy()

    new_gcn[:, moved_vertex] = np.array([neighbours[np.argmin(distance[neighbours, moved_vertex], axis=0)] for neighbours in adjacency_list])
    new_gcn[moved_vertex, moved_vertex] = moved_vertex
    changed[moved_vertex] = True

    new_gcn[moved_vertex_neighbours] = moved_vertex_neighbours_next_hop
    new_gd = gd.copy()
    new_gd[:, changed] = greedy_destination(new_gcn[:, changed].copy(), num_grs_iter)
    return new_gcn, new_gd